# Data Quality Audit — 05 · Enjoy.sa Events API

**Source:** `https://enjoy.sa/api/v1/odp/events/Get` — live official API

**Purpose in the concierge:** Events with dates, times, city, and audience restrictions.

Standardized audit covering:

```
Dataset
├── Shape
├── Columns & data types
├── Missing values
├── Duplicates
├── Invalid values
├── Outliers
├── Inconsistent categories
├── Geographic validity
├── Date/time validity
├── Data-source/license
└── Known limitations
```

> **Run where `enjoy.sa` is reachable.** This audit fetches the live API. Dates use `DD-MM-YYYY` in the `*Formatted` fields — parse with `dayfirst=True`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# Resolve repo root whether run from the audit folder or the repo root.
p = Path.cwd()
while p != p.parent and not (p / "data" / "raw").exists():
    p = p.parent
ROOT = p
print("repo root:", ROOT)

# Saudi Arabia bounding box (approx) for geographic validity checks.
SA_LAT = (16.0, 32.5)
SA_LON = (34.5, 56.0)

def iqr_outliers(series):
    """Return (count, lower, upper) of IQR outliers in a numeric series."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return 0, np.nan, np.nan
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum()), round(lo, 2), round(hi, 2)

def missing_report(df):
    m = pd.DataFrame({"missing": df.isna().sum(),
                      "missing_%": (df.isna().mean() * 100).round(1)})
    return m[m["missing"] > 0].sort_values("missing", ascending=False)

def dtype_report(df):
    return pd.DataFrame({
        "dtype": [str(t) for t in df.dtypes],
        "non_null": df.notna().sum().values,
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    }, index=df.columns)

import requests
URL = "https://enjoy.sa/api/v1/odp/events/Get"
payload = requests.get(URL, headers={"Accept":"application/json"}, timeout=60).json()
records = payload["Data"] if isinstance(payload, dict) else payload
df = pd.json_normalize(records)
print("Loaded events:", df.shape)
df.head(3)

repo root: c:\Users\bedoor.alsulami\AppData\Local\Packages\PythonSoftwareFoundation.PythonManager_3847v3x7pw1km\AppData\saudi-Digital-Concierge
Loaded events: (1, 2)


,Items,TotalCount
0,"[{'Name': ' الألعاب النارية ', 'City': None, ...",5802


## Shape

In [2]:
print("Rows:", len(df), "| Columns:", df.shape[1])

Rows: 1 | Columns: 2


## Columns & data types

In [9]:
import requests

# Refresh the live response and unwrap the Events API wrapper
resp = requests.get(URL, headers={"Accept": "application/json"}, timeout=60)
payload = resp.json()
data = payload.get("Data", payload)

if isinstance(data, dict) and "Items" in data:
    records = data["Items"]      # event list only
else:
    records = data

# Flatten the event rows into a dataframe
df = pd.json_normalize(records)

# Convert any list object values to strings so `nunique()` can run safely
for col in df.columns:
    if df[col].map(lambda x: isinstance(x, list)).any():
        df[col] = df[col].map(lambda x: str(x) if isinstance(x, list) else x)

# Now run the report
dtype_report(df)

,dtype,non_null,n_unique
Name,str,5802,4399
City,str,5798,74
StartDate,str,5802,1959
StartDateFormatted,str,5802,1832
EndDate,str,5802,2043
EndDateFormatted,str,5802,1913
StartTime,str,5802,86
StartTimeFormatted,str,5802,86
EndTime,str,5802,101
EndTimeFormatted,str,5802,101


## Missing values

In [10]:
missing_report(df) if not missing_report(df).empty else print('No missing values')

,missing,missing_%
IsMaleAllowed,41,0.7
IsFemaleAllowed,41,0.7
IsFamilyAllowed,41,0.7
City,4,0.1


## Duplicates
No unique id field — check exact rows and a business key.

In [11]:
print("Exact duplicate rows:", df.duplicated().sum())
key = [c for c in ["Name","City","StartDate","StartTime"] if c in df.columns]
print("Duplicate on", key, ":", df.duplicated(subset=key).sum())

Exact duplicate rows: 30
Duplicate on ['Name', 'City', 'StartDate', 'StartTime'] : 35


## Invalid values
Audience flags should be boolean; `EventMode` is a small controlled set.

In [12]:
for col in ["IsMaleAllowed","IsFemaleAllowed","IsFamilyAllowed"]:
    if col in df.columns:
        print(col, "->", df[col].value_counts(dropna=False).to_dict())
print("EventMode ->", df["EventMode"].value_counts(dropna=False).to_dict())

IsMaleAllowed -> {True: 5290, False: 471, None: 41}
IsFemaleAllowed -> {True: 5345, False: 416, None: 41}
IsFamilyAllowed -> {True: 5240, False: 521, None: 41}
EventMode -> {'IsExpired': 5752, 'IsActive': 50}


## Outliers
No continuous numeric measures — **N/A** (event records are categorical/temporal).

In [13]:
print("No continuous numeric columns to check for outliers.")

No continuous numeric columns to check for outliers.


## Inconsistent categories
City values (Arabic) — check spelling variants / rollups like `كل مناطق المملكة`.

In [14]:
print("Cities — %d unique" % df["City"].nunique())
print(df["City"].value_counts(dropna=False).head(20).to_string())

Cities — 74 unique
City
الرياض              3012
جدة                 1054
الطائف               214
الخبر                157
الدمام               141
كل مناطق المملكة     120
المدينة المنورة      105
الأحساء              103
أبها                  99
الخرج                 80
جازان                 60
الباحة                56
تبوك                  54
حائل                  53
الظهران               49
بريدة                 43
الجبيل                35
الدرعية               30
مكة                   29
نجران                 27


## Geographic validity
No coordinates — geography is the `City` text only (**N/A** for bbox check; map to canonical city).

In [15]:
print("Has coordinates:", any("lat" in c.lower() or "lon" in c.lower() for c in df.columns))

Has coordinates: False


## Date/time validity
Parse the `*Formatted` fields with `dayfirst=True`; verify ISO and Formatted agree and check ranges vs now.

In [16]:
sd = pd.to_datetime(df["StartDateFormatted"], dayfirst=True, errors="coerce", utc=True)
ed = pd.to_datetime(df["EndDateFormatted"], dayfirst=True, errors="coerce", utc=True)
print("Start parse fails:", sd.isna().sum(), "| range", sd.min(), "->", sd.max())
print("End parse fails:", ed.isna().sum(), "| range", ed.min(), "->", ed.max())
print("Rows where end < start:", int((ed < sd).sum()))
now = pd.Timestamp.now(tz="UTC")
print("Active (EventMode):", df["EventMode"].value_counts().to_dict())

Start parse fails: 0 | range 2017-01-05 00:00:00+00:00 -> 2026-10-01 00:00:00+00:00
End parse fails: 0 | range 2017-01-06 00:00:00+00:00 -> 2031-05-04 00:00:00+00:00
Rows where end < start: 0
Active (EventMode): {'IsExpired': 5752, 'IsActive': 50}


## Data-source / license
- **Source:** Enjoy.sa official events API.
- **License:** Official API.
- **Currency:** **live / current** — the only real-time source.

## Known limitations
- **No unique ID** → dedupe on a composite key.
- **No category field** — derive event type from `Name` if needed.
- **~99% expired** (use `EventMode` = `IsActive` for live events).
- ISO vs Formatted dates are consistent, but the Formatted fields are `DD-MM-YYYY` (parse with `dayfirst=True`).
- Arabic city names need canonical mapping; no coordinates.